# 1. Forecasting EDA

**Stage 1, Step 5.** What the demand actually looks like, and which modelling choices it forces.

The point of this notebook is not to admire the data. It is to establish four facts that decide the design:

1. **How many series there really are** — which decides whether per-series models are possible.
2. **How much of the panel is zero or near-zero** — which decides whether MAPE means anything.
3. **Where the seasonality lives** — weekly, annual, or both.
4. **How far forward the planning data extends** — which turns out to bound what can be forecast at all.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

from app.services.container import Container

repo = Container().data_repository
print("dataset version:", repo.dataset_version())

## 1. How many series?

300 products × 200 stores is 60,000 possible combinations. The number that matters is how many *actually exist*, because that is what any per-series approach would have to fit.

In [ ]:
listings = repo.execute_query(
    "SELECT product_id, store_id, COUNT(*) AS days, SUM(units) AS total_units "
    "FROM sales_daily GROUP BY 1, 2",
    max_rows=500_000,
)

products = listings["product_id"].nunique()
stores = listings["store_id"].nunique()

print(f"products                 : {products}")
print(f"stores                   : {stores}")
print(f"possible combinations    : {products * stores:,}")
print(f"real series              : {len(listings):,}")
print(f"listing density          : {len(listings) / (products * stores):.1%}")
print()
print(f"stores carrying a product: {len(listings) / products:.1f} on average")

**A product is listed in about a tenth of stores.** That has a direct consequence for sampling, and it is the reason `ml/forecasting/sampling.py` exists.

Sampling N product-store *pairs* and then passing the distinct products and distinct stores as separate filters asks the repository for the **cross product** of the two — which contains far more real series than were requested. Step 4's `build_panel` does exactly this. Let's measure the cost.

In [ ]:
from data.repositories.sampling import sample_product_store_pairs
from ml.forecasting.sampling import sample_series

for n in (50, 400, 800):
    old = sample_product_store_pairs(repo, n_pairs=n, seed=42)
    old_loaded = listings[
        listings["product_id"].isin(old["product_id"].unique())
        & listings["store_id"].isin(old["store_id"].unique())
    ]

    new = sample_series(repo, n_series=n, seed=42)
    new_loaded = listings[
        listings["product_id"].isin(new.product_ids)
        & listings["store_id"].isin(new.store_ids)
    ]

    print(f"requested {n:>4} series -> Step 4 approach loads {len(old_loaded):>5,} "
          f"({len(old_loaded)/n:>4.1f}x)   |   store-clustered loads {len(new_loaded):>5,} "
          f"({len(new_loaded)/n:.1f}x)")

Clustering the sample by store keeps the filter box tight around the pairs actually wanted, and a semi-join afterwards makes the count exact. `n_series=800` means 800.

## 2. Zero and intermittent demand

This decides which metrics are usable. MAPE is undefined at zero and unstable near it — so if a meaningful share of the panel is zero, MAPE cannot be the headline.

In [ ]:
sales = repo.get_sales(
    start_date=pd.Timestamp("2025-01-01").date(),
    end_date=pd.Timestamp("2025-12-31").date(),
    columns=["date", "product_id", "store_id", "units", "promotion_flag", "stockout_flag"],
    max_rows=5_000_000,
)

print(f"rows                 : {len(sales):,}")
print(f"zero-unit days       : {(sales['units'] == 0).mean():.2%}")
print(f"under 5 units        : {(sales['units'] < 5).mean():.2%}")
print(f"median daily units   : {sales['units'].median():.0f}")
print(f"mean daily units     : {sales['units'].mean():.1f}")
print(f"std / mean           : {sales['units'].std() / sales['units'].mean():.2f}")
print()
print(f"stockout days        : {sales['stockout_flag'].astype(bool).mean():.2%}")
print(f"promotional days     : {sales['promotion_flag'].astype(bool).mean():.2%}")

A coefficient of variation well above 1 is the important number here. Demand is drawn from an over-dispersed negative binomial, which is why Step 4 measured an **irreducible noise floor of 35% WMAPE** — the score a model knowing the *true* conditional mean would still get.

Every accuracy figure in this step should be read against that, not against zero.

## 3. Where the seasonality lives

In [ ]:
daily = sales.groupby("date", as_index=False)["units"].sum()
daily["date"] = pd.to_datetime(daily["date"])
daily["dow"] = daily["date"].dt.day_name()
daily["month"] = daily["date"].dt.month

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(daily["date"], daily["units"], linewidth=0.8)
axes[0].set_title("Total daily units, 2025")

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_dow = daily.groupby("dow")["units"].mean().reindex(order)
axes[1].bar(range(7), by_dow.to_numpy())
axes[1].set_xticks(range(7))
axes[1].set_xticklabels([d[:3] for d in order])
axes[1].set_title("Weekly cycle")

by_month = daily.groupby("month")["units"].mean()
axes[2].bar(by_month.index, by_month.to_numpy())
axes[2].set_title("Annual cycle")

plt.tight_layout()
plt.show()

weekly_swing = by_dow.max() / by_dow.min()
annual_swing = by_month.max() / by_month.min()
print(f"weekly peak/trough : {weekly_swing:.2f}x")
print(f"annual peak/trough : {annual_swing:.2f}x")

The weekly cycle is the stronger one. That is why the seasonal lag is **364 days rather than 365** — 364 is a multiple of 7, so "same day last year" is the same weekday. An off-by-one comparison against a different weekday would be worse than useless.

## 4. How far forward does the planning data go?

A forecast needs the calendar, the promotion schedule and the price plan for every day it covers. Those are `KNOWN_IN_ADVANCE` tables, so reading them forward is legitimate — but only as far as they actually extend.

In [ ]:
calendar = repo.get_calendar()
promotions = repo.get_promotions(max_rows=200_000)
pricing = repo.get_pricing(
    start_date=pd.Timestamp("2025-12-01").date(),
    end_date=pd.Timestamp("2026-06-30").date(),
    max_rows=5_000_000,
)

calendar_end = pd.to_datetime(calendar["date"]).max()
promo_end = pd.to_datetime(promotions["end_date"]).max()
price_end = pd.to_datetime(pricing["date"]).max() if not pricing.empty else None

print(f"calendar ends   : {calendar_end.date()}")
print(f"promotions end  : {promo_end.date()}")
print(f"pricing ends    : {price_end.date() if price_end is not None else 'n/a'}")
print()
for horizon in (7, 14, 30, 90):
    latest = calendar_end - pd.Timedelta(days=horizon)
    print(f"latest as-of supporting a {horizon:>2}-day horizon: {latest.date()}")

**This is a hard constraint, and it shapes the deliverable.**

A 90-day forecast is only fully informed from an as-of on or before **2025-10-02**. Past that, some forecast days have no planned promotion data at all.

The tempting response is to assume no promotion runs and carry the last price forward. That assumption has a predictable direction — promotions raise demand, so those days come back systematically low — and the resulting number is indistinguishable from a real forecast. The service therefore **refuses**, naming the latest as-of that would work.

---

## Findings

| Fact | Consequence for the design |
|---|---|
| 6,128 real series, ~10% listing density | Per-series models are infeasible; a global pooled model is the only practical choice. Sampling must be pair-exact |
| High coefficient of variation, ~35% noise floor | WMAPE is the headline; MAPE only over non-zero actuals with the exclusion count |
| Weekly cycle dominates the annual one | Seasonal lag is 364 days, not 365 |
| Planning data ends 2025-12-31 | Long horizons are refused past 2025-10-02 rather than filled with assumptions |

### Next

`02_baseline_comparison.ipynb` — what a planner achieves unaided, which is the bar any model must clear.